# V24 POC — Average slope over 30 degrees

This notebook uses the ignored real Kobo exports in `notebooks/data`, downloads a small Copernicus GLO-30 subset from OpenTopography, derives slope in a metric CRS, and evaluates the proposed `> 30°` warning.

Privacy rules: raw rows, names, phone numbers, IDs, attachment URLs, and exact coordinates are never displayed. Identifiers in outputs are one-way hashes. All generated artifacts stay below the fully ignored `notebooks/data/v24_poc_output` directory. The OpenTopography key is read from `OPENTOPOGRAPHY_API_KEY` and is never printed or persisted.

The notebook performs no database writes. It constructs unsaved instances of the existing Django `FormMetadata`, `Submission`, and `Plot` models when the project environment is available.

## 1. Environment and configuration

Run from the repository root. Required packages are `requests`, `numpy`, `rasterio`, `shapely`, and `pyproj`. For the Django model check, use the backend's project environment (Django 4.2).

In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import hashlib
import json
import math
import os
import sys
import uuid

import numpy as np
import requests
import rasterio
from pyproj import CRS, Transformer
from rasterio.transform import array_bounds, from_origin
from rasterio.warp import Resampling, reproject, transform_bounds
from shapely.geometry import Polygon, box, mapping, shape
from shapely.ops import transform as shapely_transform

def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'backend/api/v1/v1_odk/models.py').exists():
            return candidate
    raise RuntimeError('Run this notebook from inside the repository')

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / 'notebooks/data'
OUTPUT_DIR = DATA_DIR / 'v24_poc_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = next(DATA_DIR.glob('*_ab_realdata.csv'))
GEOJSON_PATH = next(DATA_DIR.glob('*_ab_realdata.geojson'))
DEM_PATH = OUTPUT_DIR / 'cop30_subset.tif'
SLOPE_PATH = OUTPUT_DIR / 'cop30_slope_degrees_utm.tif'
RESULT_PATH = OUTPUT_DIR / 'slope_results_private.json'
ALIGNED_CSV_PATH = OUTPUT_DIR / 'v24_django_aligned_sensitive.csv'
AC_DEMO_PATH = OUTPUT_DIR / 'v24_acceptance_demo.json'

OPENTOPOGRAPHY_URL = 'https://portal.opentopography.org/API/globaldem'
DEM_SOURCE = 'OpenTopography Copernicus GLO-30 DGED 2023_1'
SLOPE_THRESHOLD_DEG = 30.0
TARGET_RESOLUTION_M = 30.0
BBOX_PADDING_DEGREES = 0.01
LOW_CONFIDENCE_PIXEL_EQUIVALENT = 4.0

print({
    'csv_present': CSV_PATH.exists(),
    'geojson_present': GEOJSON_PATH.exists(),
    'cached_dem_present': DEM_PATH.exists(),
    'api_key_available': bool(os.getenv('OPENTOPOGRAPHY_API_KEY')),
})

## 2. Normalize the sensitive export without displaying it

The CSV is semicolon-delimited and contains duplicate display-label columns. This loader selects the known source column by position and produces only the fields required by the current Django extraction path. Real identifiers and names are replaced with deterministic POC values.

In [ ]:
BACKEND_DIR = REPO_ROOT / 'backend'
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from utils.polygon import (
    compute_bbox,
    coords_to_wkt,
    parse_odk_geoshape,
    validate_polygon,
)
from api.v1.v1_odk.utils.area_calc import calculate_area_ha

def hashed_id(value, prefix='poc'):
    digest = hashlib.sha256(str(value).encode('utf-8')).hexdigest()[:20]
    return f'{prefix}-{digest}'

with CSV_PATH.open(newline='', encoding='utf-8-sig') as stream:
    reader = csv.reader(stream, delimiter=';')
    header = next(reader)
    csv_rows = list(reader)

def first_column(name):
    return next(index for index, value in enumerate(header) if value == name)

COLUMN_INDEX = {
    'polygon': first_column('Auto Boundary Capture'),
    'region': first_column('Region'),
    'woreda': first_column('Name of woreda'),
    'kebele': first_column('Name of kebele'),
    'kobo_id': first_column('_id'),
    'uuid': first_column('_uuid'),
    'submission_time': first_column('_submission_time'),
}

def iso_to_epoch_ms(value):
    parsed = datetime.fromisoformat(value.replace('Z', '+00:00'))
    return int(parsed.timestamp() * 1000)

records = []
for row_number, row in enumerate(csv_rows, start=1):
    polygon_string = row[COLUMN_INDEX['polygon']].strip()
    coords = parse_odk_geoshape(polygon_string)
    valid, validation_message = validate_polygon(coords) if coords else (False, 'unparseable')
    real_uuid = row[COLUMN_INDEX['uuid']].strip()
    record = {
        'plot_ref': hashed_id(real_uuid, 'plot'),
        'submission_uuid': hashed_id(real_uuid, 'submission'),
        'kobo_id': hashed_id(row[COLUMN_INDEX['kobo_id']], 'kobo'),
        'submission_time': iso_to_epoch_ms(row[COLUMN_INDEX['submission_time']]),
        'polygon_string': polygon_string,
        'coords': coords,
        'is_valid': valid,
        'validation_message': validation_message,
        'raw_data': {
            'auto_boundary_capture': polygon_string,
            'region': 'POC region',
            'sub_region': 'POC sub-region',
            'plot_name': f'POC plot {row_number:03d}',
        },
    }
    records.append(record)

valid_records = [record for record in records if record['is_valid']]
print({
    'submission_count': len(records),
    'valid_polygon_count': len(valid_records),
    'invalid_polygon_count': len(records) - len(valid_records),
    'raw_values_displayed': False,
})

## 3. Cross-check the GeoJSON export

The file is a list of per-submission FeatureCollections rather than one standard FeatureCollection. The check below reports only aggregate structural results. The CSV ODK geoshape remains authoritative because it follows the production parser path and includes every submission.

In [ ]:
with GEOJSON_PATH.open(encoding='utf-8') as stream:
    exported_collections = json.load(stream)

geo_features = [
    collection['features'][0]
    for collection in exported_collections
    if collection.get('features')
]
geo_geometries = [shape(feature['geometry']) for feature in geo_features]
print({
    'collection_count': len(exported_collections),
    'feature_count': len(geo_features),
    'empty_collection_count': len(exported_collections) - len(geo_features),
    'valid_geometry_count': sum(geometry.is_valid for geometry in geo_geometries),
    'geometry_types': sorted(set(geometry.geom_type for geometry in geo_geometries)),
})

## 4. Download the COP30 subset from OpenTopography

The request derives one padded bounding box from all valid plots. Request parameters and exact bounds are deliberately not printed. The API key stays only in process memory. If the cached TIFF exists, no API request is made.

In [ ]:
all_coords = [coord for record in valid_records for coord in record['coords']]
private_bbox = compute_bbox(all_coords)
request_bounds = {
    'south': private_bbox['min_lat'] - BBOX_PADDING_DEGREES,
    'north': private_bbox['max_lat'] + BBOX_PADDING_DEGREES,
    'west': private_bbox['min_lon'] - BBOX_PADDING_DEGREES,
    'east': private_bbox['max_lon'] + BBOX_PADDING_DEGREES,
}

def is_tiff(content):
    return content[:4] in (b'II*\x00', b'MM\x00*')

def download_cop30(destination):
    if destination.exists() and destination.stat().st_size > 0:
        return 'cached'
    api_key = os.getenv('OPENTOPOGRAPHY_API_KEY')
    if not api_key:
        raise RuntimeError(
            'Set OPENTOPOGRAPHY_API_KEY in the notebook process environment'
        )
    params = {
        'demtype': 'COP30',
        **request_bounds,
        'outputFormat': 'GTiff',
        'API_Key': api_key,
    }
    response = requests.get(
        OPENTOPOGRAPHY_URL, params=params, timeout=(20, 300)
    )
    response.raise_for_status()
    if not is_tiff(response.content):
        raise RuntimeError('OpenTopography response was not a GeoTIFF')
    temporary = destination.with_suffix('.partial')
    temporary.write_bytes(response.content)
    temporary.replace(destination)
    return 'downloaded'

download_status = download_cop30(DEM_PATH)
with rasterio.open(DEM_PATH) as source:
    dem_metadata = {
        'download_status': download_status,
        'driver': source.driver,
        'band_count': source.count,
        'crs': str(source.crs),
        'width': source.width,
        'height': source.height,
        'source': DEM_SOURCE,
    }
print(dem_metadata)

## 5. Reproject the DEM and derive slope in degrees

Slope must not be calculated directly from longitude/latitude degrees. The COP30 subset is reprojected to the local UTM zone at 30 m, then slope is derived as `atan(sqrt((dz/dx)^2 + (dz/dy)^2))`. This is a POC implementation; production should lock and document the chosen GDAL-compatible slope algorithm.

In [ ]:
centroid_lon = (private_bbox['min_lon'] + private_bbox['max_lon']) / 2
centroid_lat = (private_bbox['min_lat'] + private_bbox['max_lat']) / 2
utm_zone = int((centroid_lon + 180) // 6) + 1
utm_epsg = (32600 if centroid_lat >= 0 else 32700) + utm_zone
utm_crs = CRS.from_epsg(utm_epsg)
to_utm = Transformer.from_crs('EPSG:4326', utm_crs, always_xy=True)

with rasterio.open(DEM_PATH) as source:
    source_data = source.read(1).astype('float32')
    source_nodata = source.nodata
    left, bottom, right, top = transform_bounds(
        source.crs, utm_crs, *source.bounds, densify_pts=21
    )
    destination_width = math.ceil((right - left) / TARGET_RESOLUTION_M)
    destination_height = math.ceil((top - bottom) / TARGET_RESOLUTION_M)
    destination_transform = from_origin(
        left, top, TARGET_RESOLUTION_M, TARGET_RESOLUTION_M
    )
    dem_utm = np.full(
        (destination_height, destination_width), np.nan, dtype='float32'
    )
    reproject(
        source=source_data,
        destination=dem_utm,
        src_transform=source.transform,
        src_crs=source.crs,
        src_nodata=source_nodata,
        dst_transform=destination_transform,
        dst_crs=utm_crs,
        dst_nodata=np.nan,
        resampling=Resampling.bilinear,
    )

dz_dy, dz_dx = np.gradient(
    dem_utm, TARGET_RESOLUTION_M, TARGET_RESOLUTION_M
)
slope_degrees = np.degrees(np.arctan(np.hypot(dz_dx, dz_dy))).astype('float32')
slope_degrees[~np.isfinite(dem_utm)] = np.nan

with rasterio.open(
    SLOPE_PATH,
    'w',
    driver='GTiff',
    height=slope_degrees.shape[0],
    width=slope_degrees.shape[1],
    count=1,
    dtype='float32',
    crs=utm_crs,
    transform=destination_transform,
    nodata=np.nan,
    compress='deflate',
) as target:
    target.write(slope_degrees, 1)

print({
    'projected_crs': utm_crs.to_string(),
    'resolution_m': TARGET_RESOLUTION_M,
    'valid_slope_cells': int(np.isfinite(slope_degrees).sum()),
    'slope_artifact_is_ignored': str(SLOPE_PATH).startswith(str(DATA_DIR)),
})

## 6. Area-weighted slope for each real plot

Most plots are smaller than one 30 × 30 m cell. The exact intersection area of each plot with every contributing slope cell is therefore used as the weight. `effective_pixel_equivalent` is plot area divided by 900 m²; values below four are marked low-resolution confidence.

In [ ]:
def build_slope_warning(mean_slope_degrees, source=DEM_SOURCE):
    if mean_slope_degrees <= SLOPE_THRESHOLD_DEG:
        return []
    return [{
        'type': 'SLOPE_TOO_STEEP',
        'severity': 'warning',
        'note': (
            f'Average slope is {mean_slope_degrees:.2f} degrees '
            f'(threshold: > {SLOPE_THRESHOLD_DEG:.1f} degrees). '
            f'Dataset source: {source}.'
        ),
    }]

def pixel_polygon(transform, row, col):
    x0, y0 = transform * (col, row)
    x1, y1 = transform * (col + 1, row + 1)
    return box(min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1))

def clamp(value, minimum, maximum):
    return max(minimum, min(maximum, value))

def weighted_slope_for_polygon(polygon_utm):
    inverse = ~destination_transform
    min_col, max_row = inverse * (polygon_utm.bounds[0], polygon_utm.bounds[1])
    max_col, min_row = inverse * (polygon_utm.bounds[2], polygon_utm.bounds[3])
    row_start = clamp(math.floor(min_row) - 1, 0, slope_degrees.shape[0])
    row_stop = clamp(math.ceil(max_row) + 1, 0, slope_degrees.shape[0])
    col_start = clamp(math.floor(min_col) - 1, 0, slope_degrees.shape[1])
    col_stop = clamp(math.ceil(max_col) + 1, 0, slope_degrees.shape[1])
    weighted_total = 0.0
    covered_area = 0.0
    contributing_cells = 0
    for row in range(row_start, row_stop):
        for col in range(col_start, col_stop):
            value = float(slope_degrees[row, col])
            if not math.isfinite(value):
                continue
            intersection_area = polygon_utm.intersection(
                pixel_polygon(destination_transform, row, col)
            ).area
            if intersection_area <= 0:
                continue
            contributing_cells += 1
            covered_area += intersection_area
            weighted_total += value * intersection_area
    if covered_area <= 0:
        return None
    return {
        'mean_slope_degrees': weighted_total / covered_area,
        'contributing_cells': contributing_cells,
        'coverage_percent': min(100.0, covered_area / polygon_utm.area * 100.0),
        'effective_pixel_equivalent': polygon_utm.area / (TARGET_RESOLUTION_M ** 2),
    }

slope_results = []
for record in valid_records:
    polygon_wgs84 = Polygon(record['coords'])
    polygon_utm = shapely_transform(to_utm.transform, polygon_wgs84)
    result = weighted_slope_for_polygon(polygon_utm)
    if result is None:
        slope_results.append({
            'plot_ref': record['plot_ref'],
            'status': 'unavailable',
        })
        continue
    mean_slope = result['mean_slope_degrees']
    slope_results.append({
        'plot_ref': record['plot_ref'],
        'status': 'complete',
        'mean_slope_degrees': round(mean_slope, 4),
        'flagged_over_30_degrees': mean_slope > SLOPE_THRESHOLD_DEG,
        'contributing_cells': result['contributing_cells'],
        'coverage_percent': round(result['coverage_percent'], 2),
        'effective_pixel_equivalent': round(result['effective_pixel_equivalent'], 3),
        'low_resolution_confidence': (
            result['effective_pixel_equivalent'] < LOW_CONFIDENCE_PIXEL_EQUIVALENT
        ),
        'source': DEM_SOURCE,
        'resolution_m': TARGET_RESOLUTION_M,
    })

complete_results = [r for r in slope_results if r['status'] == 'complete']
safe_summary = {
    'total_submissions': len(records),
    'valid_polygons': len(valid_records),
    'slope_complete': len(complete_results),
    'slope_unavailable': len(slope_results) - len(complete_results),
    'flagged_over_30_degrees': sum(r['flagged_over_30_degrees'] for r in complete_results),
    'low_resolution_confidence': sum(r['low_resolution_confidence'] for r in complete_results),
    'source': DEM_SOURCE,
    'threshold_degrees': SLOPE_THRESHOLD_DEG,
}
RESULT_PATH.write_text(
    json.dumps({'summary': safe_summary, 'plots': slope_results}, indent=2),
    encoding='utf-8',
)

result_by_plot = {result['plot_ref']: result for result in slope_results}
aligned_fieldnames = [
    'form_asset_uid', 'submission_uuid', 'kobo_id', 'submission_time',
    'submitted_by', 'instance_name', 'approval_status', 'raw_data_json',
    'system_data_json', 'plot_uuid', 'plot_name', 'polygon_source_field',
    'polygon_wkt', 'min_lat', 'max_lat', 'min_lon', 'max_lon', 'region',
    'sub_region', 'created_at', 'area_ha', 'flagged_for_review',
    'flagged_reason_json', 'proposed_geospatial_metrics_json',
]
with ALIGNED_CSV_PATH.open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=aligned_fieldnames)
    writer.writeheader()
    for record in valid_records:
        slope_result = result_by_plot[record['plot_ref']]
        bbox = compute_bbox(record['coords'])
        warning = build_slope_warning(
            slope_result['mean_slope_degrees']
        ) if slope_result['status'] == 'complete' else []
        metric = {
            'slope': {
                'status': slope_result['status'],
                'value': slope_result.get('mean_slope_degrees'),
                'unit': 'degrees',
                'source': DEM_SOURCE,
                'source_version': 'DGED-2023_1',
                'details': {
                    'contributing_cells': slope_result.get('contributing_cells'),
                    'coverage_percent': slope_result.get('coverage_percent'),
                    'effective_pixel_equivalent': slope_result.get('effective_pixel_equivalent'),
                    'low_resolution_confidence': slope_result.get('low_resolution_confidence'),
                },
            }
        }
        writer.writerow({
            'form_asset_uid': 'v24-poc-unsaved',
            'submission_uuid': record['submission_uuid'],
            'kobo_id': record['kobo_id'],
            'submission_time': record['submission_time'],
            'submitted_by': 'poc-enumerator',
            'instance_name': record['raw_data']['plot_name'],
            'approval_status': '',
            'raw_data_json': json.dumps(record['raw_data']),
            'system_data_json': '{}',
            'plot_uuid': record['plot_ref'],
            'plot_name': record['raw_data']['plot_name'],
            'polygon_source_field': 'auto_boundary_capture',
            'polygon_wkt': coords_to_wkt(record['coords']),
            'min_lat': bbox['min_lat'],
            'max_lat': bbox['max_lat'],
            'min_lon': bbox['min_lon'],
            'max_lon': bbox['max_lon'],
            'region': record['raw_data']['region'],
            'sub_region': record['raw_data']['sub_region'],
            'created_at': record['submission_time'],
            'area_ha': calculate_area_ha(record['polygon_string']),
            'flagged_for_review': bool(warning),
            'flagged_reason_json': json.dumps(warning),
            'proposed_geospatial_metrics_json': json.dumps(metric),
        })

print({**safe_summary, 'model_aligned_rows_written': len(valid_records)})

## 7. Acceptance criteria demo using existing records

The actual GLO-30 results contain no plot over 30 degrees, so they cannot naturally demonstrate the positive threshold branch. This deterministic contract demo reuses two existing hashed plot records and their real measured values, but substitutes controlled evaluation values of `30.01` and `30.00` degrees. The controlled values are labelled as demo inputs and never replace the observed terrain measurements.

The generated `v24_acceptance_demo.json` demonstrates that `30.01` is flagged, exactly `30.00` is not flagged, and the positive warning contains the measured value, strict threshold, and dataset source. The file remains under the ignored POC output directory.

In [ ]:
if len(complete_results) < 2:
    raise RuntimeError('At least two complete real results are required for the AC demo')

controlled_scenarios = [
    ('above_threshold', complete_results[0], 30.01),
    ('exactly_at_threshold', complete_results[1], 30.00),
]
acceptance_demo_rows = []
for scenario, real_result, controlled_value in controlled_scenarios:
    warning = build_slope_warning(controlled_value)
    acceptance_demo_rows.append({
        'scenario': scenario,
        'uses_existing_hashed_plot': True,
        'plot_ref': real_result['plot_ref'],
        'observed_mean_slope_degrees': real_result['mean_slope_degrees'],
        'controlled_demo_mean_slope_degrees': controlled_value,
        'controlled_demo_input': True,
        'flagged_for_review': bool(warning),
        'flagged_reason': warning,
        'geospatial_metrics': {
            'slope': {
                'status': 'complete',
                'value': controlled_value,
                'unit': 'degrees',
                'source': DEM_SOURCE,
                'source_version': 'DGED-2023_1',
            }
        },
    })

above_row, exact_row = acceptance_demo_rows
above_note = above_row['flagged_reason'][0]['note']
acceptance_checks = {
    'ac_above_30_is_flagged': above_row['flagged_for_review'] is True,
    'ac_exactly_30_is_not_flagged': exact_row['flagged_for_review'] is False,
    'ac_warning_has_measured_value': '30.01 degrees' in above_note,
    'ac_warning_has_strict_threshold': 'threshold: > 30.0 degrees' in above_note,
    'ac_warning_has_dataset_source': DEM_SOURCE in above_note,
}
assert all(acceptance_checks.values()), acceptance_checks
AC_DEMO_PATH.write_text(
    json.dumps({
        'acceptance_checks': acceptance_checks,
        'demo_rows': acceptance_demo_rows,
    }, indent=2),
    encoding='utf-8',
)
print(json.dumps({
    'acceptance_checks': acceptance_checks,
    'demo_output': [{
        'scenario': row['scenario'],
        'mean_slope_degrees': row['controlled_demo_mean_slope_degrees'],
        'flagged_for_review': row['flagged_for_review'],
        'warning': row['flagged_reason'],
    } for row in acceptance_demo_rows],
}, indent=2))

## 8. Existing Django model compatibility — no database writes

This cell constructs unsaved instances using the existing model fields and production `extract_plot_data` helper. It deliberately does not call `.save()`. The proposed `geospatial_metrics` field does not exist until V24 is implemented, so slope results remain in the separate POC structure above. If project setup is unavailable in the current kernel, run this cell inside the backend container/environment.

In [ ]:
def validate_unsaved_django_instances(records_to_check):
    os.environ.setdefault(
        'DJANGO_SETTINGS_MODULE', 'african_bamboo_dashboard.settings'
    )
    import django
    django.setup()
    from api.v1.v1_odk.models import FormMetadata, Plot, Submission
    from utils.polygon import extract_plot_data

    form = FormMetadata(
        asset_uid='v24-poc-unsaved',
        name='V24 slope POC',
        polygon_field='auto_boundary_capture',
        region_field='region',
        sub_region_field='sub_region',
        plot_name_field='plot_name',
    )
    failures = []
    for record in records_to_check:
        extracted = extract_plot_data(record['raw_data'], form)
        submission = Submission(
            uuid=record['submission_uuid'],
            form=form,
            kobo_id=record['kobo_id'],
            submission_time=record['submission_time'],
            submitted_by='poc-enumerator',
            instance_name=record['raw_data']['plot_name'],
            raw_data=record['raw_data'],
            system_data={},
            approval_status=None,
        )
        plot = Plot(
            uuid=record['plot_ref'],
            form=form,
            submission=submission,
            plot_name=extracted['plot_name'],
            polygon_source_field=extracted['polygon_source_field'],
            polygon_wkt=extracted['polygon_wkt'],
            min_lat=extracted['min_lat'],
            max_lat=extracted['max_lat'],
            min_lon=extracted['min_lon'],
            max_lon=extracted['max_lon'],
            region=extracted['region'],
            sub_region=extracted['sub_region'],
            created_at=record['submission_time'],
            area_ha=calculate_area_ha(record['polygon_string']),
        )
        try:
            submission.full_clean(
                exclude=['form', 'updated_by'], validate_unique=False
            )
            plot.full_clean(
                exclude=['form', 'submission', 'farmer'], validate_unique=False
            )
        except Exception as error:
            failures.append(type(error).__name__)
    return {
        'checked': len(records_to_check),
        'valid_unsaved_model_pairs': len(records_to_check) - len(failures),
        'failure_count': len(failures),
        'database_writes': 0,
    }

try:
    django_model_summary = validate_unsaved_django_instances(valid_records)
except Exception as setup_error:
    django_model_summary = {
        'status': 'environment_unavailable',
        'error_type': type(setup_error).__name__,
        'database_writes': 0,
    }
print(django_model_summary)

## 9. Synthetic example — what a flagged plot looks like

No real plot exceeds 30 degrees; the observed range is 0.30-20.69. The positive
branch therefore has no visual evidence behind it. This section hand-draws a
25 m example plot on a **public escarpment far from the collection area**,
derives slope there with the same formula used in section 5, and evaluates it
with the same `build_slope_warning` rule used on the real data.

Two properties make this safe to commit: the polygon is fabricated, and the
location is a named landmark rather than a farm, so the map discloses nothing
about where African Bamboo works. Terrain comes from the keyless Copernicus
GLO-30 mirror on AWS Open Data, so no OpenTopography key is needed here.

In [ ]:
sys.path.insert(0, str(REPO_ROOT / 'notebooks'))
from poc_common import (
    DEMO_LOCATIONS,
    preview_map,
    public_dem_url,
    synthetic_square,
)

DEMO_SIDE_M = 25.0
demo_lon, demo_lat, demo_place = DEMO_LOCATIONS['steep_slope']
demo_plot = synthetic_square(demo_lon, demo_lat, DEMO_SIDE_M)
demo_utm_crs = CRS.from_epsg(
    32600 + int((demo_lon + 180) // 6) + 1
)
demo_to_utm = Transformer.from_crs('EPSG:4326', demo_utm_crs, always_xy=True)
demo_plot_utm = shapely_transform(demo_to_utm.transform, demo_plot)

# Same pipeline as section 5, over a small window at the demo location:
# reproject to a metric grid, then slope = atan(sqrt((dz/dx)^2 + (dz/dy)^2)).
with rasterio.open(public_dem_url(demo_lon, demo_lat)) as demo_source:
    pad = 0.01
    window = rasterio.windows.from_bounds(
        demo_lon - pad, demo_lat - pad, demo_lon + pad, demo_lat + pad,
        transform=demo_source.transform,
    )
    demo_dem = demo_source.read(1, window=window).astype('float32')
    demo_src_transform = demo_source.window_transform(window)
    demo_left, demo_bottom, demo_right, demo_top = transform_bounds(
        demo_source.crs, demo_utm_crs,
        *rasterio.windows.bounds(window, demo_source.transform),
        densify_pts=21,
    )

demo_width = math.ceil((demo_right - demo_left) / TARGET_RESOLUTION_M)
demo_height = math.ceil((demo_top - demo_bottom) / TARGET_RESOLUTION_M)
demo_transform = from_origin(
    demo_left, demo_top, TARGET_RESOLUTION_M, TARGET_RESOLUTION_M
)
demo_utm_dem = np.full((demo_height, demo_width), np.nan, dtype='float32')
reproject(
    source=demo_dem,
    destination=demo_utm_dem,
    src_transform=demo_src_transform,
    src_crs='EPSG:4326',
    dst_transform=demo_transform,
    dst_crs=demo_utm_crs,
    resampling=Resampling.bilinear,
    dst_nodata=np.nan,
)
demo_dzdy, demo_dzdx = np.gradient(demo_utm_dem, TARGET_RESOLUTION_M)
demo_slope = np.degrees(
    np.arctan(np.sqrt(demo_dzdx ** 2 + demo_dzdy ** 2))
)

demo_values = []
demo_weights = []
inverse = ~demo_transform
min_col, max_row = inverse * (demo_plot_utm.bounds[0], demo_plot_utm.bounds[1])
max_col, min_row = inverse * (demo_plot_utm.bounds[2], demo_plot_utm.bounds[3])
for row in range(max(int(min_row) - 1, 0),
                 min(int(max_row) + 2, demo_slope.shape[0])):
    for col in range(max(int(min_col) - 1, 0),
                     min(int(max_col) + 2, demo_slope.shape[1])):
        value = demo_slope[row, col]
        if np.isnan(value):
            continue
        piece = pixel_polygon(demo_transform, row, col).intersection(
            demo_plot_utm
        )
        if piece.is_empty or piece.area <= 0:
            continue
        demo_values.append(float(value))
        demo_weights.append(piece.area)

demo_mean_slope = float(np.average(demo_values, weights=demo_weights))
demo_warning = build_slope_warning(demo_mean_slope)
print(json.dumps({
    'location': demo_place,
    'mean_slope_degrees': round(demo_mean_slope, 2),
    'threshold_degrees': SLOPE_THRESHOLD_DEG,
    'contributing_cells': len(demo_values),
    'flagged': bool(demo_warning),
    'note': demo_warning[0]['note'] if demo_warning else None,
}, indent=2))

The map below is the visual check. A red outline means the rule
fired; click the polygon for the measured values. Run
`pip install -r notebooks/requirements.txt` if folium is missing.

In [ ]:
preview_map(
    demo_plot,
    flagged=bool(demo_warning),
    title=f'V24 example - {demo_place}',
    rows={
        'Mean slope': f'{demo_mean_slope:.2f} degrees',
        'Threshold': f'> {SLOPE_THRESHOLD_DEG:.1f} degrees',
        'Result': 'SLOPE_TOO_STEEP' if demo_warning else 'pass',
        'Contributing cells': len(demo_values),
        'Source': DEM_SOURCE,
    },
    zoom=16,
)

## 10. Publish the reviewable summary

Output is split by sensitivity.

`notebooks/outputs/` is **tracked by git** and receives only the aggregate
summary and the acceptance checks. `notebooks/data/v24_poc_output/` stays
**fully ignored**: it holds the per-plot rows, the DEM and derived slope
rasters — whose georeferenced bounds are the padded plot bounding box — and
`v24_django_aligned_sensitive.csv`, which contains exact polygon geometry.

In [ ]:
sys.path.insert(0, str(REPO_ROOT / 'notebooks'))
from poc_common import write_public_output

public = write_public_output('v24_slope_summary.json', {
    'poc': 'v24',
    'metric': 'slope',
    'notebook': 'notebooks/v24_average_slope_poc.ipynb',
    'dataset': {
        'source': DEM_SOURCE,
        'source_version': 'DGED-2023_1',
        'source_provider': 'OpenTopography',
        'target_resolution_m': TARGET_RESOLUTION_M,
    },
    'summary': safe_summary,
    'observed_mean_slope_degrees': {
        'min': round(min(r['mean_slope_degrees'] for r in complete_results), 2),
        'median': round(sorted(
            r['mean_slope_degrees'] for r in complete_results
        )[len(complete_results) // 2], 2),
        'max': round(max(r['mean_slope_degrees'] for r in complete_results), 2),
    },
    'acceptance_checks': acceptance_checks,
    'boundary_demo': [
        {
            'scenario': row['scenario'],
            'controlled_demo_mean_slope_degrees': (
                row['controlled_demo_mean_slope_degrees']
            ),
            'flagged_for_review': row['flagged_for_review'],
            'note': (
                row['flagged_reason'][0]['note']
                if row['flagged_reason'] else None
            ),
        }
        for row in acceptance_demo_rows
    ],
})

print({
    'ignored_private_files': [
        RESULT_PATH.name, AC_DEMO_PATH.name, ALIGNED_CSV_PATH.name,
        DEM_PATH.name, SLOPE_PATH.name,
    ],
    'tracked_public_file': public['file'],
})

## 11. Review checklist and interpretation

The POC is technically successful when all ODK geoshapes parse, the DEM covers the plots, slope results are deterministic, unavailable calculations are explicit, and unsaved model instances match current Django constraints.

The key product/GIS decision remains resolution: a large share of real plots are smaller than one 30 m cell. For these records the number is a GLO-30 location-level terrain estimate, not a detailed measurement of slope variation inside the farm boundary. Review the aggregate `low_resolution_confidence` count before approving the 30° rule.

For an independent QGIS spot-check, load the ignored `cop30_slope_degrees_utm.tif` locally and compare several hashed records. Do not copy the real polygons or raster outside the ignored POC directory.

Citation: European Space Agency (2024), Copernicus Global Digital Elevation Model, distributed by OpenTopography, https://doi.org/10.5069/G9028PQB.